In [ ]:
import pandas as pd
import numpy as np 
from matplotlib import pyplot as plt

from pathlib import Path
from time import strftime

from keras.layers import Flatten , Dense  , Embedding , Input , Concatenate , Dropout , BatchNormalization 
from keras.layers import   LeakyReLU
from keras.optimizers import Adam
from keras.callbacks import EarlyStopping , TensorBoard
from keras.losses import BinaryCrossentropy
from keras.models import Model
from tensorflow.random import set_seed

In [ ]:
x_train = pd.read_csv('./datas/Out_Stage3/x_train')
x_valid = pd.read_csv('./datas/Out_Stage3/x_valid')
y_train = pd.read_csv('./datas/Out_Stage3/y_train')
y_valid = pd.read_csv('./datas/Out_Stage3/y_valid')
numeric_cols = ['PAY_0','PAY_2','PAY_3','PAY_4','PAY_5','PAY_6','AGE','BILL_AMT1','BILL_AMT2',
                'BILL_AMT3','BILL_AMT4','BILL_AMT5','BILL_AMT6','PAY_AMT1','PAY_AMT2','PAY_AMT3','PAY_AMT4','PAY_AMT5','PAY_AMT6']

set_seed(42)

In [ ]:
def get_model(reg_params):
     input_marriage = Input(shape=(1,) , name='marriage')
     embeding_marriage = Embedding(4,4 // 2 ,name = 'marriage_emb')(input_marriage)
     layer_marriage = Flatten()(embeding_marriage)

     input_education = Input(shape=(1,) , name='education')
     embeding_education = Embedding(7,7 // 2 ,name = 'education_emb')(input_education)
     layer_education = Flatten()(embeding_education)

     input_sex = Input(shape=(1,) , name='sex')
     embeding_sex = Embedding(2,2 // 2 ,name = 'sex_emb')(input_sex)
     layer_sex = Flatten()(embeding_sex)

     numeric_input = Input(shape=(19,))
     layer_numeric = Dense(32, activation='relu')(numeric_input)


     concat = Concatenate()([layer_marriage,layer_sex,layer_education ,layer_numeric])

     output = Dense(1,activation='sigmoid')

     tuned_params = {
          "n_neurons" : 64,
          "l2_rate" : 1.4792092042502516e-05,
          "activation" : "relu",
          "n_dropout" : 0.4,
          "n_hidden" : 5
     }

     


     layer_main = Dense(tuned_params['n_neurons'])(concat)
     layer_main = BatchNormalization()(layer_main)
     layer_main = LeakyReLU()(layer_main)
     layer_main = Dropout(reg_params['dropout_rate'])(layer_main)

     for _ in range(tuned_params['n_hidden']-1):
          layer_main = Dense(tuned_params['n_neurons'])(layer_main)
          layer_main = BatchNormalization()(layer_main)
          layer_main =  LeakyReLU()(layer_main)
          layer_main = Dropout(reg_params['dropout_rate'])(layer_main)


     layer_output = output(layer_main)

     model_tuned = Model(inputs=[input_sex, input_marriage , input_education , numeric_input] , outputs =[layer_output])

     model_tuned.compile(optimizer=Adam(learning_rate=1e-3 , weight_decay=reg_params['l2_rate']) , loss=BinaryCrossentropy() ,metrics=['accuracy'])

     return model_tuned

def fit_model(params ,epo , calls):
     model_tuned = get_model(params)
     tune_regularization_model_h = model_tuned.fit(
    [x_train['SEX'],x_train['MARRIAGE'],x_train['EDUCATION'],x_train[numeric_cols]]
     , y_train  , epochs=epo ,
       validation_data=(
           [x_valid['SEX'],x_valid['MARRIAGE'],x_valid['EDUCATION'],x_valid[numeric_cols]]
           ,y_valid) ,
           callbacks=calls
             )
     
     return tune_regularization_model_h.history['val_loss'] , tune_regularization_model_h.history['val_accuracy']

In [ ]:
def get_persent_of_pos(losses):

    losses=np.array(losses)

    sub_loses = losses[:-1] - losses[1:]

    sub_loses = (sub_loses > 0).astype(int)

    return float(np.count_nonzero(sub_loses)/len(sub_loses))



In [ ]:
params_datas = [
    {"l2_rate" : 0.0000147,"dropout_rate" : 0.4,},
    {"l2_rate" : 0.0000147,"dropout_rate" : 0.45,},
    {"l2_rate" : 0.0000147,"dropout_rate" : 0.5,},
    {"l2_rate" : 0.0000147,"dropout_rate" : 0.55,},
    {"l2_rate" : 0.0000147,"dropout_rate" : 0.6,},

    {"l2_rate" : 0.0001,"dropout_rate" : 0.4,},
    {"l2_rate" : 0.0001,"dropout_rate" : 0.45,},
    {"l2_rate" : 0.00008,"dropout_rate" : 0.5,},
    {"l2_rate" : 0.00008,"dropout_rate" : 0.55,},
    {"l2_rate" : 0.00006,"dropout_rate" : 0.6,},
]


In [ ]:

all_losses =[]
all_accurs = []
for i,p in enumerate(params_datas):
    losses , acuurs = fit_model(p , 15 ,[])
    all_losses.append(losses)
    all_accurs.append(acuurs)
    #plt.close()
    #plt.title(i),
    #plt.plot(losses)
    #plt.show()
    #print(get_persent_of_pos(losses))

In [ ]:

fig , axes =plt.subplots(11, 2,figsize=(16, 40))
axes = axes.ravel()

for i in range(len(all_losses)):


    axes[(2*i)-2].set_xlabel(str(i))
    axes[(2*i)-2].plot(all_losses[i])
    axes[(2*i)-2].set_ylim(0.44,0.475)

    axes[(2*i)-1].set_xlabel(str(i))
    axes[(2*i)-1].plot(all_accurs[i],color=(1,0,0))
    axes[(2*i)-1].set_ylim(0.805,0.825)

    axes[20].plot(all_losses[i])
    axes[20].set_ylim(0.44,0.475)

    axes[21].plot(all_accurs[i],color=(1,0,0))
    axes[21].set_ylim(0.805,0.825)


In [ ]:
best_reg_params = params_datas[2]

In [ ]:

def get_run_logdir(root_logdir="my_logs"):
    return Path(root_logdir) / strftime("run_%Y_%m_%d_%H_%M_%S")


earlyStopping = EarlyStopping('val_loss' ,patience=3)
tensorboard_cb = TensorBoard(get_run_logdir() , profile_batch=(100,200))

fit_model(best_reg_params , 20 , [earlyStopping,tensorboard_cb])